# 05 Time Resolution

Determine time-dependece of RUNX-factor binding. 

## 05.01 Initialization

Load tools and initialize working dirtectory.

In [2]:
docker_run() {
    docker run --rm -i \
        -u $(id -u):$(id -g) \
        -v /home/dalbao:/home/dalbao \
        -v /etc/timezone:/etc/timezone:ro \
        -v /etc/localtime:/etc/localtime:ro \
        -w $(pwd) \
        "$@"
}

# Define software to use:
## deeptools for analysis and visualization of deep-sequencing data
deeptools() {
    docker_run quay.io/biocontainers/deeptools:3.5.6--pyhdfd78af_0 "$@"
}
deeptools plotHeatmap --version

# Define software to use:
## bedtools for bed file manipulation
bedtools() {
    docker_run staphb/bedtools:2.31.1 bedtools "$@"
}
bedtools --version

cd /home/dalbao/AlbaoRunx3Manuscript/cutnrun
mkdir -p 05_time_resolved/source

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-jw58hqcj because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
plotHeatmap 3.5.6
bedtools v2.31.1


In [3]:
# Softlink raw peaks into source if does not exist in source
# Raw Peaks
for f in source_data/260713_reBAM2_noDeDup_CPM/03_peak_calling/04_called_peaks/macs2/*narrowPeak; do
    if [ ! -f "05_time_resolved/source/$(basename "$f")" ]; then
        ln -s "../../$f" "05_time_resolved/source/$(basename "$f")"
    else
        echo "File $(basename "$f") exists in source."
    fi
done

# Above Inflection
for f in 02_peakqc/*above*.bed; do
    if [ ! -f "05_time_resolved/source/$(basename "$f")" ]; then
        ln -s "../../$f" "05_time_resolved/source/$(basename "$f")"
    else
        echo "File $(basename "$f") exists in source."
    fi
done

rm 05_time_resolved/source/*H3K*

# 05.02 Redefine Peaks

First make a "liberal" set of genes defined by peaks with `qValue < 1e-5` [`-log10(qValue) > 5`]

In [6]:
# For all .narrowPeak files in 05_time_resolved/source,
# Keep only rows where $9 > 5 (qValue)
# keep only columns 1-3, 7 and 9 (chrom start end signal qValue)
# Then rename to remove _R1.macs2_peaks.narrowPeak from the filename and replace with .liberal.bed
# Save in folder 05_time_resolved/new_bed

mkdir -p 05_time_resolved/liberal_bed

for f in 05_time_resolved/source/*.narrowPeak; do
    base=$(basename "$f")
    # strip the full _R1.macs2_peaks.narrowPeak suffix, not just .narrowPeak
    out="${base/_R1.macs2_peaks.narrowPeak/.liberal.bed}"
    # BEGIN{OFS="\t"} forces tab output -> valid BED for bedtools/deeptools/UCSC
    awk 'BEGIN{OFS="\t"} $9 > 5 {print $1, $2, $3, $7, $9}' "$f" > "05_time_resolved/liberal_bed/$out"
done

# 05.03 Overlap UpSet Plots

Determine peak overlaps across conditions using `bedtools multiinter`, then visualize as UpSet plots in R (`UpSetR`). Three plots: Runx1 across conditions, Runx3 across conditions, and a combined plot across all 12 condition/factor sets.

Run for both the liberal peak set (`qValue`-filtered, from 05.02) and the stringent peak set (pre-filtered, above-inflection, `05_time_resolved/source/*.merged.macs2_peaks.above_inflection.bed`).

First, check R environment for required packages.

In [7]:
Rscript -e '
cat("R version:", R.version.string, "\n")
pkgs <- c("UpSetR", "ggplot2")
for (p in pkgs) {
  cat(p, "installed:", requireNamespace(p, quietly = TRUE), "\n")
}
cat("Library paths:\n")
print(.libPaths())
cat("Writable library path:", any(file.access(.libPaths(), mode = 2) == 0), "\n")
'


R version: R version 4.4.2 (2024-10-31) 
UpSetR installed: FALSE 
ggplot2 installed: TRUE 
Library paths:
[1] "/home/dalbao/R/x86_64-conda-linux-gnu-library/4.4"
[2] "/opt/conda/lib/R/library"                         
Writable library path: TRUE 



In [22]:
# Build genomic overlap matrices with bedtools multiinter:
# for each merged interval across the input beds, a 0/1 column per condition
# marking whether that condition has a peak there. This is exactly the
# binary set-membership format UpSetR expects.
#
# Runs for both the liberal peak set (qValue-filtered) and the stringent
# peak set (pre-filtered, above-inflection).

mkdir -p 05_time_resolved/upset/sorted 05_time_resolved/upset/plots

# multiinter requires sorted input
for f in 05_time_resolved/liberal_bed/*.liberal.bed; do
    bedtools sort -i "$f" > "05_time_resolved/upset/sorted/$(basename "$f")"
done
for f in 05_time_resolved/source/*.merged.macs2_peaks.above_inflection.bed; do
    base="$(basename "$f")"
    name="${base%.merged.macs2_peaks.above_inflection.bed}"
    bedtools sort -i "$f" > "05_time_resolved/upset/sorted/${name}.stringent.bed"
done

# Chronological/logical condition order (used consistently across all plots)
conditions="early late terminal memory shCd19 shRunx3"

for peakset in liberal stringent; do
    suffix="${peakset}.bed"

    # Runx1 across conditions
    bedtools multiinter -header \
        -i $(for c in $conditions; do echo "05_time_resolved/upset/sorted/${c}_Runx1.${suffix}"; done) \
        -names $conditions \
        > "05_time_resolved/upset/${peakset}_runx1_multiinter.txt"

    # Runx3 across conditions
    bedtools multiinter -header \
        -i $(for c in $conditions; do echo "05_time_resolved/upset/sorted/${c}_Runx3.${suffix}"; done) \
        -names $conditions \
        > "05_time_resolved/upset/${peakset}_runx3_multiinter.txt"

    # Combined: all 12 factor/condition sets
    combined_files=""
    combined_names=""
    for c in $conditions; do
        for tf in Runx1 Runx3; do
            combined_files="$combined_files 05_time_resolved/upset/sorted/${c}_${tf}.${suffix}"
            combined_names="$combined_names ${c}_${tf}"
        done
    done
    bedtools multiinter -header -i $combined_files -names $combined_names \
        > "05_time_resolved/upset/${peakset}_combined_multiinter.txt"
done

wc -l 05_time_resolved/upset/*multiinter.txt

  167475 05_time_resolved/upset/combined_multiinter.txt
  167475 05_time_resolved/upset/liberal_combined_multiinter.txt
   63498 05_time_resolved/upset/liberal_runx1_multiinter.txt
  100010 05_time_resolved/upset/liberal_runx3_multiinter.txt
   63498 05_time_resolved/upset/runx1_multiinter.txt
  100010 05_time_resolved/upset/runx3_multiinter.txt
   25259 05_time_resolved/upset/stringent_combined_multiinter.txt
   14214 05_time_resolved/upset/stringent_runx1_multiinter.txt
   10358 05_time_resolved/upset/stringent_runx3_multiinter.txt
  711797 total


Plot each overlap matrix as an UpSet plot using `05_time_resolved/upset_plot.R` (UpSetR). PDFs are written to `05_time_resolved/upset/plots/`.

In [23]:
for peakset in liberal stringent; do
    label="$(echo "${peakset:0:1}" | tr '[:lower:]' '[:upper:]')${peakset:1}"

    Rscript 05_time_resolved/upset_plot.R \
        "05_time_resolved/upset/${peakset}_runx1_multiinter.txt" \
        "05_time_resolved/upset/plots/${peakset}_runx1_upset.pdf" \
        "Runx1 (${label})"

    Rscript 05_time_resolved/upset_plot.R \
        "05_time_resolved/upset/${peakset}_runx3_multiinter.txt" \
        "05_time_resolved/upset/plots/${peakset}_runx3_upset.pdf" \
        "Runx3 (${label})"

    Rscript 05_time_resolved/upset_plot.R \
        "05_time_resolved/upset/${peakset}_combined_multiinter.txt" \
        "05_time_resolved/upset/plots/${peakset}_combined_upset.pdf" \
        "Runx1 + Runx3 (${label})" \
        30
done

Wrote 05_time_resolved/upset/plots/liberal_runx1_upset.pdf 
Wrote 05_time_resolved/upset/plots/liberal_runx3_upset.pdf 
Wrote 05_time_resolved/upset/plots/liberal_combined_upset.pdf 
Wrote 05_time_resolved/upset/plots/stringent_runx1_upset.pdf 
Wrote 05_time_resolved/upset/plots/stringent_runx3_upset.pdf 
Wrote 05_time_resolved/upset/plots/stringent_combined_upset.pdf 

